# Climate Data – A hands-on python course
Author: Pedro Herrera Lormendez (pedrolormendez@gmail.com)

**Updated 2025:** Enhanced with WMO baseline verification, statistical significance testing, and uncertainty quantification

## Computing Climatologies and Anomalies

### What is a climatology?

A **climatology** represents the long-term average state of the climate system over a specified reference period. In climate science:

* **Standard period:** 30 years (WMO recommendation)
* **Purpose:** Establish a baseline for comparing current conditions
* **Statistical basis:** Long enough to average out interannual variability but short enough to be relevant

### WMO Standard Baseline Periods

The World Meteorological Organization (WMO) recommends specific 30-year periods:

* **1961-1990:** Historical baseline (pre-acceleration warming)
* **1981-2010:** Transitional period
* **1991-2020:** Current standard (updated 2021)

**Important:** The choice of baseline affects anomaly magnitudes!

### What is a climate anomaly?

An **anomaly** is the deviation from the climatological mean:

$$\text{Anomaly} = \text{Observed Value} - \text{Climatological Mean}$$

Anomalies are preferred over absolute values because:
1. Remove systematic biases between datasets
2. Highlight changes relative to normal conditions
3. Facilitate spatial comparisons
4. Reduce impact of local variations

### Applications

* Identifying climate trends
* Detecting extreme events
* Model evaluation and bias correction
* Agricultural planning and risk assessment
* Climate change attribution

### Useful Visualizations:
* [Global temperature distribution](https://climvis.org/content/anim/ltm/globe/t2m_globe_1991-2020_ltm/t2m_globe_1991-2020_ltm.html)
* [Global precipitation rate](https://climvis.org/content/anim/ltm/globe/tp_globe_1991-2020_ltm/tp_globe_1991-2020_ltm.html)
* [P - E (Precipitation minus Evaporation)](https://climvis.org/content/anim/ltm/globe/pme_globe_1991-2020_ltm/pme_globe_1991-2020_ltm.html)
* [More animations](https://climvis.org/animations.html)

### Datasets used in this notebook

We use **ERA5 reanalysis** monthly data for Europe:

* **2m temperature:** [Download t2m file](https://drive.google.com/file/d/1-JZirUHXP7sDGIUoFG_h8znRn2rfKT8m/view?usp=sharing)
* **Precipitation:** [Download rainfall file](https://drive.google.com/file/d/1ErVv5A0DNhDQKvmVYbUNUAIGysV-p6sd/view?usp=sharing)

And **E-OBS gridded observations** for validation:
* High-quality observational dataset
* 0.25° × 0.25° resolution
* Daily data from 1950 to present
* Ideal for Europe-focused studies

### Importing necessary modules

In [ ]:
import sys
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
import pandas as pd
import warnings

sys.path.append(os.path.abspath('../help_code'))
import tools

# Configure plotting
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore', category=RuntimeWarning)

---
## Part 1: Temperature Climatology and Anomalies

### Computing seasonal mean climatology for temperature

In [ ]:
# Reading the ERA5 temperature file
file_path = '../data/t2m_monthly_era5.nc'

try:
    DS = xr.open_dataset(file_path)
    DS = tools.convert_and_sort_coords(DS)
    print(f"✓ Dataset loaded successfully")
    print(f"\nDataset information:")
    print(DS)
except FileNotFoundError:
    print(f"Error: File not found. Please download from the link above.")
    raise

In [ ]:
# Extract temperature variable and convert to °C
t2m = DS['t2m'] - 273.15
t2m.attrs['units'] = '°C'
t2m.attrs['long_name'] = '2 metre temperature'

print(f"Time range: {pd.to_datetime(t2m.time[0].values).year} to {pd.to_datetime(t2m.time[-1].values).year}")
print(f"Number of months: {len(t2m.time)}")
print(f"Spatial extent: {float(t2m.latitude.min())}°N to {float(t2m.latitude.max())}°N")
print(f"                {float(t2m.longitude.min())}°E to {float(t2m.longitude.max())}°E")

### WMO Standard Baseline Period: 1991-2020

**Important considerations for seasonal climatology:**

When computing seasonal means, we must be careful about incomplete seasons:
* **Winter (DJF):** December-January-February
* **Spring (MAM):** March-April-May
* **Summer (JJA):** June-July-August
* **Autumn (SON):** September-October-November

To avoid incomplete winters:
* **Start:** March 1991 (skip Jan-Feb 1991, as Dec 1990 is missing)
* **End:** February 2021 (includes complete DJF 2020/2021)

This ensures all seasons have complete 3-month data for the 30-year period.

**Reference:** WMO Technical Regulations (WMO-No. 49)

In [ ]:
# Extract data for the 1991-2020 climatological period (Mar 1991 - Feb 2021)
t2m_clim_period = t2m.sel(time=slice('1991-03', '2021-02'))

print(f"Climatology period: {t2m_clim_period.time[0].values} to {t2m_clim_period.time[-1].values}")
print(f"Number of months: {len(t2m_clim_period.time)}")
print(f"Expected: 360 months (30 years × 12 months)")
print(f"Actual: {len(t2m_clim_period.time)} months")

# Verify we have complete seasons
if len(t2m_clim_period.time) == 360:
    print("\n✓ Complete 30-year climatological period (WMO standard)")
else:
    print(f"\n⚠ Warning: Period length is {len(t2m_clim_period.time)} instead of 360 months")

### Seasonal aggregation

Using xarray's [groupby()](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.groupby.html) to compute seasonal means.

**Important:** XArray assigns seasons based on the starting month:
* **DJF:** December-January-February
* **MAM:** March-April-May  
* **JJA:** June-July-August
* **SON:** September-October-November

In [ ]:
# Compute seasonal means for the climatology period
t2m_seasonal_clim = t2m_clim_period.groupby('time.season').mean(dim='time')

print(f"Seasonal climatology computed")
print(f"Dimensions: {t2m_seasonal_clim.dims}")
print(f"Seasons: {t2m_seasonal_clim.season.values}")

# Reorder seasons to conventional sequence: DJF, MAM, JJA, SON
season_order = ['DJF', 'MAM', 'JJA', 'SON']
t2m_seasonal_clim = t2m_seasonal_clim.reindex(season=season_order)
print(f"\n✓ Seasons reordered: {list(t2m_seasonal_clim.season.values)}")

### Visualizing the seasonal climatology

In [ ]:
# Create 4-panel plot with enhanced cartography
fig = plt.figure(figsize=(16, 12))

seasons = ['DJF', 'MAM', 'JJA', 'SON']
season_names = ['Winter (DJF)', 'Spring (MAM)', 'Summer (JJA)', 'Autumn (SON)']

for i, (season, season_name) in enumerate(zip(seasons, season_names)):
    ax = fig.add_subplot(2, 2, i+1, projection=ccrs.PlateCarree())
    
    # Plot data
    im = t2m_seasonal_clim.sel(season=season).plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        vmin=-15, vmax=30,
        cmap='RdYlBu_r',
        add_colorbar=True,
        cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8, 'extend': 'both'}
    )
    
    # Add geographic features
    ax.coastlines(resolution='50m', linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')
    ax.add_feature(cfeature.LAKES, alpha=0.3)
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    ax.set_title(f'{season_name}', fontsize=12, fontweight='bold')

plt.suptitle('Seasonal Temperature Climatology (1991-2020)\nERA5 Reanalysis', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('seasonal_climatology_temperature.png', dpi=300, bbox_inches='tight')
plt.show()

### Practice time

<div style="background-color:lightgreen; padding:15px">
<b>Exercise: European mean seasonal temperature</b><br><br>
Compute and visualize the seasonal temperature cycle for Europe.
<ul>
    <li>Crop to European domain: 35-70°N, -10 to 35°E</li>
    <li>Compute the spatial mean using <code>DataArray.mean(dim=('latitude', 'longitude'))</code></li>
    <li>Plot the seasonal cycle with appropriate labels</li>
    <li>Add error bars showing spatial standard deviation</li>
</ul>
</div>

In [ ]:
# Crop data to European domain
t2m_seasonal_clim_eu = t2m_seasonal_clim.sel(latitude=slice(35, 70), longitude=slice(-10, 35))

# Compute spatial mean and standard deviation
t2m_eu_mean = t2m_seasonal_clim_eu.mean(dim=('latitude', 'longitude'))
t2m_eu_std = t2m_seasonal_clim_eu.std(dim=('latitude', 'longitude'))

# Create figure
plt.figure(figsize=(12, 6))

# Plot with error bars
seasons_x = np.arange(len(season_order))
plt.errorbar(seasons_x, t2m_eu_mean.values, yerr=t2m_eu_std.values,
             marker='o', markersize=10, linewidth=2, capsize=8, capthick=2,
             color='darkblue', ecolor='lightblue', label='Mean ± 1σ')

plt.xticks(seasons_x, season_order)
plt.xlabel('Season', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.title('European Seasonal Mean Temperature (1991-2020 Climatology)\nERA5 Reanalysis', 
          fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('european_seasonal_cycle.png', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics
print("European Seasonal Statistics (1991-2020):")
print("=" * 50)
for season in season_order:
    mean_val = float(t2m_eu_mean.sel(season=season))
    std_val = float(t2m_eu_std.sel(season=season))
    print(f"{season}: {mean_val:6.2f}°C ± {std_val:5.2f}°C")
print(f"\nAnnual range: {float(t2m_eu_mean.max() - t2m_eu_mean.min()):.2f}°C")

---
## Computing Temperature Anomalies with Statistical Significance

### Understanding anomalies

An anomaly quantifies how much a value deviates from the climatological normal:

$$\text{Anomaly}(t) = \text{Observation}(t) - \text{Climatology}$$

**Key questions for each anomaly:**
1. Is it **statistically significant**?
2. What is the **confidence interval**?
3. How does it compare to **natural variability**?

### Yearly temperature anomalies

We'll compute anomalies relative to the **1961-1990 baseline** (historical standard) and compare with **1991-2020** (current standard).

In [ ]:
# Compute yearly means for the entire dataset
t2m_yearly = t2m.groupby('time.year').mean(dim='time')

# Compute climatologies for two baseline periods
# Historical baseline: 1961-1990 (WMO standard)
t2m_clim_1961_1990 = t2m_yearly.sel(year=slice(1961, 1990)).mean(dim='year')
t2m_std_1961_1990 = t2m_yearly.sel(year=slice(1961, 1990)).std(dim='year')

# Current baseline: 1991-2020 (WMO current standard)
t2m_clim_1991_2020 = t2m_yearly.sel(year=slice(1991, 2020)).mean(dim='year')
t2m_std_1991_2020 = t2m_yearly.sel(year=slice(1991, 2020)).std(dim='year')

print("Climatologies computed:")
print(f"  1961-1990: Mean = {float(t2m_clim_1961_1990.mean()):.2f}°C, Std = {float(t2m_std_1961_1990.mean()):.2f}°C")
print(f"  1991-2020: Mean = {float(t2m_clim_1991_2020.mean()):.2f}°C, Std = {float(t2m_std_1991_2020.mean()):.2f}°C")
print(f"\nBaseline shift: {float((t2m_clim_1991_2020 - t2m_clim_1961_1990).mean()):.2f}°C")
print("This reflects warming between the two periods!")

In [ ]:
# Compute anomalies relative to 1961-1990
t2m_anomaly_1961_1990 = t2m_yearly - t2m_clim_1961_1990

# Compute anomalies relative to 1991-2020
t2m_anomaly_1991_2020 = t2m_yearly - t2m_clim_1991_2020

print("✓ Anomalies computed relative to both baselines")
print(f"\nAnomaly dimensions: {t2m_anomaly_1961_1990.dims}")
print(f"Anomaly shape: {t2m_anomaly_1961_1990.shape}")

### Statistical significance testing for anomalies

**Question:** Is a given anomaly significantly different from the baseline variability?

**Method:** One-sample t-test

**Null hypothesis:** The anomaly is not significantly different from zero (i.e., within natural variability)

**Test statistic:**
$$t = \frac{\text{Anomaly}}{\text{SE}} = \frac{\text{Anomaly}}{\sigma / \sqrt{n}}$$

Where:
* $\sigma$ = standard deviation of the climatological period
* $n$ = number of years in climatology (30)
* SE = standard error of the mean

**Decision rule:** Reject null hypothesis if p < 0.05 (95% confidence)

In [ ]:
# Compute European spatial mean for anomalies
t2m_anomaly_eu = t2m_anomaly_1961_1990.sel(latitude=slice(35, 70), longitude=slice(-10, 35)).mean(dim=('latitude', 'longitude'))

# Baseline statistics
baseline_years = t2m_yearly.sel(year=slice(1961, 1990), latitude=slice(35, 70), longitude=slice(-10, 35))
baseline_mean_eu = baseline_years.mean(dim=('latitude', 'longitude'))
baseline_std = float(baseline_mean_eu.std(dim='year'))
baseline_n = 30
se = baseline_std / np.sqrt(baseline_n)

# Compute t-statistics and p-values for each year
t_stats = t2m_anomaly_eu / se
from scipy.stats import t as t_dist
dof = baseline_n - 1
p_values = 2 * (1 - t_dist.cdf(np.abs(t_stats), dof))  # Two-tailed test

# Compute 95% confidence intervals
t_crit = t_dist.ppf(0.975, dof)
ci_lower = t2m_anomaly_eu - t_crit * se
ci_upper = t2m_anomaly_eu + t_crit * se

print("Statistical Significance Analysis")
print("=" * 70)
print(f"Baseline: 1961-1990")
print(f"Baseline std dev: {baseline_std:.3f}°C")
print(f"Standard error: {se:.3f}°C")
print(f"Critical t-value (95% CI): ±{t_crit:.3f}")
print()

# Show recent years
print("Recent years (2015-2022):")
print(f"{'Year':<6} {'Anomaly':<10} {'t-stat':<10} {'p-value':<10} {'Significant?':<12}")
print("-" * 70)
for year in range(2015, 2023):
    if year in t2m_anomaly_eu.year.values:
        anom = float(t2m_anomaly_eu.sel(year=year))
        t_stat = float(t_stats.sel(year=year))
        p_val = float(p_values.sel(year=year))
        sig = "Yes ***" if p_val < 0.001 else "Yes **" if p_val < 0.01 else "Yes *" if p_val < 0.05 else "No"
        print(f"{year:<6} {anom:>8.3f}°C {t_stat:>8.3f} {p_val:>10.4f} {sig:<12}")

# Count significant anomalies
significant_count = (p_values < 0.05).sum().values
total_years = len(p_values)
print("\n" + "=" * 70)
print(f"Significant anomalies (p < 0.05): {significant_count} out of {total_years} years")
print(f"Percentage: {100 * significant_count / total_years:.1f}%")

### Visualizing anomalies with confidence intervals

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

years = t2m_anomaly_eu.year.values
anomalies = t2m_anomaly_eu.values

# Color bars by significance
colors = ['red' if p < 0.05 else 'gray' for p in p_values.values]

# Plot bars
bars = ax.bar(years, anomalies, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)

# Add confidence interval bounds
ax.fill_between(years, ci_lower.values, ci_upper.values, 
                 alpha=0.2, color='blue', label='95% Confidence Interval')

# Add reference lines
ax.axhline(y=0, color='black', linewidth=1.5, linestyle='-', label='Baseline (1961-1990)')
ax.axhline(y=baseline_std, color='orange', linewidth=1, linestyle='--', alpha=0.7, label='±1σ natural variability')
ax.axhline(y=-baseline_std, color='orange', linewidth=1, linestyle='--', alpha=0.7)

# Add 10-year rolling mean
anomaly_smooth = t2m_anomaly_eu.rolling(year=10, center=True).mean()
ax.plot(years, anomaly_smooth.values, color='darkred', linewidth=3, label='10-year mean')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Temperature Anomaly (°C)', fontsize=12)
ax.set_title('European Annual Temperature Anomalies (Relative to 1961-1990)\n' +
             'Red bars indicate statistically significant anomalies (p < 0.05)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('european_temperature_anomalies_significance.png', dpi=300, bbox_inches='tight')
plt.show()

### Spatial anomaly patterns with significance masks

In [ ]:
# Select recent years for comparison
years_to_plot = [2000, 2010, 2018, 2020]

fig = plt.figure(figsize=(18, 10))

for i, year in enumerate(years_to_plot):
    ax = fig.add_subplot(2, 2, i+1, projection=ccrs.PlateCarree())
    
    # Get anomaly for this year
    anom = t2m_anomaly_1961_1990.sel(year=year)
    
    # Compute t-statistics spatially
    t_stat_spatial = anom / (t2m_std_1961_1990 / np.sqrt(30))
    
    # Significance mask (p < 0.05, corresponds to |t| > 2.045 for df=29)
    significant = np.abs(t_stat_spatial) > t_crit
    
    # Plot anomaly
    im = anom.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        vmin=-3, vmax=3,
        cmap='RdBu_r',
        add_colorbar=True,
        cbar_kwargs={'label': 'Anomaly (°C)', 'shrink': 0.8, 'extend': 'both'}
    )
    
    # Overlay significance stippling
    lons, lats = np.meshgrid(anom.longitude, anom.latitude)
    ax.scatter(lons[significant], lats[significant], 
               c='none', s=1, marker='.', edgecolors='black', linewidths=0.3,
               transform=ccrs.PlateCarree(), alpha=0.5)
    
    # Add features
    ax.coastlines(resolution='50m', linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    ax.set_title(f'{year}', fontsize=12, fontweight='bold')

plt.suptitle('Annual Temperature Anomalies (Relative to 1961-1990)\n' +
             'Stippling indicates statistical significance (p < 0.05)',
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('spatial_anomalies_with_significance.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Part 2: Precipitation Climatology and Anomalies

### Important difference for precipitation

Unlike temperature (which is averaged), **precipitation must be summed** over the period, then averaged:

1. **Sum** daily/monthly precipitation to get seasonal/annual totals
2. **Average** these totals over the climatological period

This is because precipitation is a **flux** (amount per time), not a **state variable**.

### Using E-OBS gridded observations

**E-OBS dataset:**
* High-quality observational gridded dataset
* Daily precipitation from rain gauge observations
* 0.25° × 0.25° resolution for Europe
* Ideal for validation and regional studies

**Download:** [E-OBS daily rainfall dataset](https://knmi-ecad-assets-prd.s3.amazonaws.com/ensembles/data/Grid_0.25deg_reg_ensemble/rr_ens_mean_0.25deg_reg_v28.0e.nc)

**Full datasets:** https://surfobs.climate.copernicus.eu/dataaccess/access_eobs.php#datafiles

In [ ]:
# Reading E-OBS daily precipitation file
file_path_rr = '../data/rr_ens_mean_0.25deg_reg_v28.0e.nc'

try:
    DS_rr = xr.open_dataset(file_path_rr)
    DS_rr = tools.convert_and_sort_coords(DS_rr)
    rr = DS_rr.rr
    
    print("✓ E-OBS precipitation data loaded")
    print(f"\nDataset information:")
    print(rr)
    print(f"\nTime range: {pd.to_datetime(rr.time[0].values).date()} to {pd.to_datetime(rr.time[-1].values).date()}")
    print(f"Number of days: {len(rr.time)}")
    print(f"Units: {rr.attrs.get('units', 'Not specified')}")
except FileNotFoundError:
    print("Error: E-OBS file not found. Please download from the link above.")
    raise

### Computing seasonal precipitation climatology

**Method:**
1. Select the climatological period (1991-2020)
2. **Resample** to quarterly (seasonal) sums using 'QS-DEC'
3. **Group by season** and compute mean

**'QS-DEC' meaning:**
* QS = Quarter Start
* DEC = Starting in December
* Results in: DJF, MAM, JJA, SON

In [ ]:
# Select 1991-2020 period (Mar 1991 - Feb 2021 for complete seasons)
rr_clim_period = rr.sel(time=slice('1991-03', '2021-02'))

print(f"Climatology period: {rr_clim_period.time[0].values} to {rr_clim_period.time[-1].values}")
print(f"Number of days: {len(rr_clim_period.time)}")

# Step 1: Resample to seasonal sums
rr_seasonal_sums = rr_clim_period.resample(time='QS-DEC').sum(dim='time')
print(f"\n✓ Seasonal sums computed: {len(rr_seasonal_sums.time)} seasons")
print(f"Expected: 120 seasons (30 years × 4 seasons)")

# Step 2: Average over the climatological period
rr_seasonal_clim = rr_seasonal_sums.groupby('time.season').mean(dim='time')

# Reorder seasons
rr_seasonal_clim = rr_seasonal_clim.reindex(season=season_order)

print(f"\n✓ Seasonal precipitation climatology (1991-2020) computed")
print(f"Dimensions: {rr_seasonal_clim.dims}")

### Visualizing seasonal precipitation climatology

In [ ]:
fig = plt.figure(figsize=(16, 12))

for i, (season, season_name) in enumerate(zip(seasons, season_names)):
    ax = fig.add_subplot(2, 2, i+1, projection=ccrs.PlateCarree())
    
    # Plot data
    im = rr_seasonal_clim.sel(season=season).plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        vmin=0, vmax=600,
        cmap='Blues',
        add_colorbar=True,
        cbar_kwargs={'label': 'Precipitation (mm)', 'shrink': 0.8, 'extend': 'max'}
    )
    
    # Add features
    ax.coastlines(resolution='50m', linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')
    ax.add_feature(cfeature.LAKES, alpha=0.3)
    ax.add_feature(cfeature.RIVERS, alpha=0.3, linewidth=0.5)
    
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    ax.set_title(f'{season_name}', fontsize=12, fontweight='bold')

plt.suptitle('Seasonal Precipitation Climatology (1991-2020)\nE-OBS Gridded Observations', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('seasonal_climatology_precipitation.png', dpi=300, bbox_inches='tight')
plt.show()

### Computing yearly precipitation anomalies

For precipitation anomalies:
* Often expressed as **percentage** rather than absolute difference
* Or as **standardized anomalies** (in units of standard deviation)

In [ ]:
# Compute yearly total precipitation
rr_yearly = rr.groupby('time.year').sum(dim='time')

# Compute 1961-1990 climatology
rr_clim_1961_1990 = rr_yearly.sel(year=slice(1961, 1990)).mean(dim='year')
rr_std_1961_1990 = rr_yearly.sel(year=slice(1961, 1990)).std(dim='year')

# Compute absolute anomalies (mm)
rr_anomaly_abs = rr_yearly - rr_clim_1961_1990

# Compute percentage anomalies
rr_anomaly_pct = 100 * (rr_yearly - rr_clim_1961_1990) / rr_clim_1961_1990

# Compute standardized anomalies (in units of σ)
rr_anomaly_std = (rr_yearly - rr_clim_1961_1990) / rr_std_1961_1990

print("✓ Precipitation anomalies computed (absolute, percentage, and standardized)")
print(f"\nEuropean mean climatology (1961-1990): {float(rr_clim_1961_1990.sel(latitude=slice(35,70), longitude=slice(-10,35)).mean()):.0f} mm/year")

### Visualizing precipitation anomalies

In [ ]:
# European mean precipitation anomalies
rr_anomaly_eu_abs = rr_anomaly_abs.sel(latitude=slice(35, 70), longitude=slice(-10, 35)).mean(dim=('latitude', 'longitude'))
rr_anomaly_eu_pct = rr_anomaly_pct.sel(latitude=slice(35, 70), longitude=slice(-10, 35)).mean(dim=('latitude', 'longitude'))

# Baseline variability
baseline_rr = rr_yearly.sel(year=slice(1961, 1990), latitude=slice(35, 70), longitude=slice(-10, 35)).mean(dim=('latitude', 'longitude'))
baseline_rr_std = float(baseline_rr.std(dim='year'))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

years = rr_anomaly_eu_abs.year.values

# Plot 1: Absolute anomalies
colors_precip = ['blue' if val > 0 else 'brown' for val in rr_anomaly_eu_abs.values]
ax1.bar(years, rr_anomaly_eu_abs.values, color=colors_precip, alpha=0.7, edgecolor='black', linewidth=0.5)
ax1.axhline(y=0, color='black', linewidth=1.5)
ax1.axhline(y=baseline_rr_std, color='gray', linewidth=1, linestyle='--', alpha=0.7, label='±1σ')
ax1.axhline(y=-baseline_rr_std, color='gray', linewidth=1, linestyle='--', alpha=0.7)
ax1.plot(years, rr_anomaly_eu_abs.rolling(year=10, center=True).mean().values, 
         color='darkblue', linewidth=3, label='10-year mean')
ax1.set_ylabel('Precipitation Anomaly (mm/year)', fontsize=11)
ax1.set_title('European Annual Precipitation Anomalies (Relative to 1961-1990)', 
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Percentage anomalies
ax2.bar(years, rr_anomaly_eu_pct.values, color=colors_precip, alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0, color='black', linewidth=1.5)
ax2.plot(years, rr_anomaly_eu_pct.rolling(year=10, center=True).mean().values, 
         color='darkblue', linewidth=3, label='10-year mean')
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('Precipitation Anomaly (%)', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('european_precipitation_anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nRecent decade (2013-2022) mean anomaly: {float(rr_anomaly_eu_abs.sel(year=slice(2013,2022)).mean()):.1f} mm/year")
print(f"Recent decade (2013-2022) mean anomaly: {float(rr_anomaly_eu_pct.sel(year=slice(2013,2022)).mean()):.1f}%")

---
## Physical Plausibility Checks

Always validate your climatology and anomaly calculations:

1. **Magnitude checks:** Are values within expected ranges?
2. **Spatial patterns:** Do they match known climate features?
3. **Temporal consistency:** Are trends physically plausible?
4. **Mass/energy conservation:** Do anomalies balance globally?
5. **Comparison with literature:** Do results align with published studies?

In [ ]:
print("Physical Plausibility Checks")
print("=" * 70)

# Temperature checks
print("\n1. TEMPERATURE CLIMATOLOGY (1991-2020):")
print("   European seasonal range:")
for season in season_order:
    temp = float(t2m_seasonal_clim_eu.sel(season=season).mean())
    print(f"     {season}: {temp:6.2f}°C")
annual_range = float(t2m_seasonal_clim_eu.max() - t2m_seasonal_clim_eu.min())
print(f"   Annual temperature range: {annual_range:.2f}°C")
if 15 < annual_range < 30:
    print("   ✓ Plausible for mid-latitude continental climate")
else:
    print(f"   ⚠ Warning: Annual range seems unusual")

# Temperature trend check
print("\n2. TEMPERATURE TREND (1940-2022):")
years_full = t2m_yearly.sel(latitude=slice(35,70), longitude=slice(-10,35)).mean(dim=('latitude','longitude'))
slope, _, _, p_value, _ = stats.linregress(years_full.year.values, years_full.values)
trend_per_decade = slope * 10
print(f"   European trend: {trend_per_decade:.3f}°C/decade (p = {p_value:.2e})")
if 0.1 < trend_per_decade < 0.5:
    print("   ✓ Consistent with observed global warming rates")
else:
    print(f"   ⚠ Warning: Trend outside typical range (0.1-0.5°C/decade)")

# Precipitation checks
print("\n3. PRECIPITATION CLIMATOLOGY (1991-2020):")
print("   European seasonal totals:")
rr_seasonal_clim_eu = rr_seasonal_clim.sel(latitude=slice(35,70), longitude=slice(-10,35))
for season in season_order:
    precip = float(rr_seasonal_clim_eu.sel(season=season).mean())
    print(f"     {season}: {precip:6.0f} mm")
annual_precip = float(rr_clim_1961_1990.sel(latitude=slice(35,70), longitude=slice(-10,35)).mean())
print(f"   Annual total (1961-1990): {annual_precip:.0f} mm")
if 400 < annual_precip < 1200:
    print("   ✓ Plausible for European average")
else:
    print(f"   ⚠ Warning: Annual precipitation seems unusual")

# Anomaly checks
print("\n4. RECENT WARMING (2011-2020 vs 1961-1990):")
recent_anom = float(t2m_anomaly_eu.sel(year=slice(2011, 2020)).mean())
print(f"   Mean anomaly: {recent_anom:.2f}°C")
if 0.5 < recent_anom < 2.5:
    print("   ✓ Consistent with IPCC AR6 findings")
    print("   ✓ Matches observed European warming")
else:
    print(f"   ⚠ Warning: Anomaly outside expected range")

print("\n" + "=" * 70)
print("Validation complete. Results appear physically plausible.")

---
## Summary and Key Takeaways

### What we learned:

**1. WMO Standard Baselines:**
* 30-year periods are standard (1961-1990, 1991-2020)
* Baseline choice affects anomaly magnitudes
* Always document which baseline you use

**2. Proper Climatology Calculation:**
* Temperature: **Average** over time
* Precipitation: **Sum** then average
* Handle incomplete seasons carefully

**3. Statistical Significance:**
* Use t-tests for anomaly significance
* Report confidence intervals
* Account for natural variability
* Distinguish statistical vs practical significance

**4. Uncertainty Quantification:**
* Standard errors reflect sampling uncertainty
* Confidence intervals show plausible ranges
* Spatial variability matters

**5. Physical Validation:**
* Always check plausibility
* Compare with literature
* Verify spatial patterns
* Question unexpected results

### Best Practices:

✅ **DO:**
* Use WMO standard 30-year periods
* Document baseline period clearly
* Test statistical significance
* Report uncertainty estimates
* Validate against observations
* Check physical plausibility

❌ **DON'T:**
* Use arbitrary baseline periods
* Ignore statistical significance
* Neglect uncertainty
* Average precipitation inappropriately
* Skip validation checks

### Scientific Context:

**Temperature:**
* Europe has warmed faster than global average
* Recent decade shows significant warming (p < 0.001)
* Trends are spatially heterogeneous (Arctic amplification)

**Precipitation:**
* More variable than temperature
* Trends less clear and regionally dependent
* Mediterranean drying vs northern wetting

### References:

* WMO Technical Regulations (WMO-No. 49)
* IPCC AR6 WG1 (2021): Climate Change 2021
* ERA5: Hersbach et al. (2020), QJRMS
* E-OBS: Cornes et al. (2018), JGR Atmospheres

### Next Steps:
* **Notebook 5:** Future climate projections (CMIP6)
* **Notebook 6:** Climate attribution and extreme events